In [1]:
# 리뷰 데이터 전처리
from tensorflow.keras.datasets import imdb
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
# 최대 단어 개수 만개로 제한하고 훈련 데이터와 테스트 데이터 받아옴
vocab_size = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


- 훈련 데이터와 이에 대한 레이블이 각각 X_train, y_train에 테스트 데이터와 이에 대한 레이블이 각각 X_test, y_test에 저장됨.
- IMDB 리뷰 데이터는 이미 정수 인코딩이 된 상태므로 남은 전처리는 패딩뿐임
- 리뷰의 최대 길이와 평균 길이 확인

In [3]:
print('리뷰의 최대길이: {}'.format(max(len(l) for l in X_train)))

리뷰의 최대길이: 2494


In [4]:
print('리뷰의 평균길이: {}'.format(sum(map(len, X_train))/len(X_train)))

리뷰의 평균길이: 238.71364


In [5]:
# 평균 길이보다 두배 정도로 데이터를 패딩해, 범위 넘어가면 버림 조치
max_len = 500
X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

In [6]:
X_train

array([[   0,    0,    0, ...,   19,  178,   32],
       [   0,    0,    0, ...,   16,  145,   95],
       [   0,    0,    0, ...,    7,  129,  113],
       ...,
       [   0,    0,    0, ...,    4, 3586,    2],
       [   0,    0,    0, ...,   12,    9,   23],
       [   0,    0,    0, ...,  204,  131,    9]], dtype=int32)

# 바다나우 어텐션

In [7]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Embedding, Bidirectional, LSTM, Concatenate, Dropout
from tensorflow.keras import Model, Input
from tensorflow.keras import optimizers

In [8]:
class BahdanauAttention(Model):
  def __init__(self, units):
    super(BahdanauAttention, self).__init__()
    self.W1 = Dense(units)
    self.W2 = Dense(units)
    self.V = Dense(1)

# 양방향으로 이어지는 경우가 많아서 , call같은 함수형으로 사용
  def call(self, values, query):
    hidden_with_time_axis = tf.expand_dims(query, 1)
    score = self.V(tf.nn.tanh(self.W1(values) + self.W2(hidden_with_time_axis)))
    attention_weights = tf.nn.softmax(score, axis=1)
    context_vector = attention_weights * values
    context_vector = tf.reduce_sum(context_vector, axis=1)

    return context_vector, attention_weights

In [9]:
# 모델 설계, 임베딩(벡터화)
sequence_input = Input(shape=(max_len,), dtype='int32')
embedded_sequences = Embedding(vocab_size, 128, input_length=max_len, mask_zero=True)(sequence_input)

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [10]:
lstm = Bidirectional(LSTM(64, dropout=0.5, return_sequences=True))(embedded_sequences)

In [11]:
lstm, forward_h, forward_c, backward_h, backward_c = Bidirectional(LSTM(64, dropout=0.5, return_sequences=True, return_state=True))(lstm)

In [12]:
print(lstm.shape, forward_h.shape, forward_c.shape, backward_h.shape, backward_c.shape)

(None, 500, 128) (None, 64) (None, 64) (None, 64) (None, 64)


- 순방향 LSTM의 은닉 상태와 셀상태를 forward_h, forward_c에 저장

- 역방향 LSTM의 은닉 상태와 셀 상태를 backward_h, backward_c에 저장


- 각 은닉 상태나 셀 상태의 경우에는 128차원을 가지는데, lstm의 경우에는 (500 × 128)의 크기를 가짐. foward 방향과 backward 방향이 연결된 hidden state벡터가 모든 시점에 대해서 존재함을 의미함

- 양방향 LSTM을 사용할 경우에는 순방향 LSTM과 역방향 LSTM 각각 은닉 상태와 셀 상태를 가지므로, 양방향 LSTM의 은닉 상태와 셀 상태를 사용하려면 두 방향의 LSTM의 상태들을 연결(concatenate)함

In [13]:
state_h = Concatenate()([forward_h, backward_h]) # 은닉 상태
state_c = Concatenate()([forward_c, backward_c]) # 셀 상태

In [14]:
# 은닉 상태를 입력으로 해 컨텍스트 벡터를 획득

attention = BahdanauAttention(64) # 가중치 크기 정의
context_vector, attention_weights = attention(lstm, state_h)

/usr/local/lib/python3.10/dist-packages/keras/src/layers/layer.py:915: UserWarning: Layer 'bahdanau_attention' (of type BahdanauAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [15]:
# 컨텍스트 벡터를 밀집층에 통과, 이진 분류이므로 최종 출력층에 1개 뉴런 배치하고, 활성화 함수로 시그모이드 함수를 이용

dense1 = Dense(20, activation="relu")(context_vector)
dropout = Dropout(0.5)(dense1)
output = Dense(1, activation="sigmoid")(dropout)
model = Model(inputs=sequence_input, outputs=output)

In [16]:
# 옵티마이저로 아담 옵티마이저 사용 후 모델을 컴파일
# 시그모이드 함수(0또는1)를 사용해 손실함수로 binary_crossentropy를 사용

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [17]:
# 검증 데이터로 테스트 데이터 사용해 에포크 끝날 때마다 테스트 데이터에 대한 정확도 출력

history = model.fit(X_train, y_train, epochs = 3, batch_size = 256, validation_data=(X_test, y_test), verbose=1)

Epoch 1/3
98/98 ━━━━━━━━━━━━━━━━━━━━ 1077s 11s/step - accuracy: 0.6321 - loss: 0.6196 - val_accuracy: 0.8547 - val_loss: 0.3357
Epoch 2/3
98/98 ━━━━━━━━━━━━━━━━━━━━ 1086s 11s/step - accuracy: 0.8982 - loss: 0.2791 - val_accuracy: 0.8866 - val_loss: 0.2822
Epoch 3/3
98/98 ━━━━━━━━━━━━━━━━━━━━ 1098s 11s/step - accuracy: 0.9264 - loss: 0.2111 - val_accuracy: 0.8867 - val_loss: 0.2964


In [18]:
print("\n 테스트 정확도: %.4f" % (model.evaluate(X_test, y_test)[1]))

782/782 ━━━━━━━━━━━━━━━━━━━━ 293s 375ms/step - accuracy: 0.8881 - loss: 0.2985

 테스트 정확도: 0.8867


In [19]:
import numpy as np

T, H = 5, 4
hs = np.random.randn(T, H)
a = np.array([0.8, 0.1, 0.03, 0.05, 0.02])

ar = a.reshape(5, 1).repeat(4, axis=1)
print(ar.shape)

t = hs * ar
print(t.shape)

c = np.sum(t, axis=0)
print(c.shape)

(5, 4)
(5, 4)
(4,)
